# Role C — Improvement Method 1: Pruning (M1)

**Câu hỏi thí nghiệm:** Một cây nhỏ hơn nhiều lần có giữ được hiệu năng dự báo hay không?

Notebook này xây dựng M1 bằng cost-complexity pruning, sau đó bổ sung grid bắt buộc cho `max_depth` và `min_samples_leaf`. Toàn bộ siêu tham số được chọn bằng cross-validation **chỉ trên train set**. Held-out test set được niêm phong cho đến khi cấu hình M1 đã được khóa, rồi chỉ được đánh giá qua helper dùng chung `evaluate_model()`.

Tài liệu API: [cost-complexity pruning](https://scikit-learn.org/1.8/auto_examples/tree/plot_cost_complexity_pruning.html), [GridSearchCV](https://scikit-learn.org/1.8/modules/generated/sklearn.model_selection.GridSearchCV.html), và [StratifiedKFold](https://scikit-learn.org/1.8/modules/generated/sklearn.model_selection.StratifiedKFold.html).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier


def find_repo_root(start: Path | None = None) -> Path:
    """Find the nearest ancestor containing the shared data pipeline."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy repo root chứa src/data.py")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import get_train_test
from src.evaluate import RESULT_COLUMNS, evaluate_model
from src.visualize import plot_tree_figure

RANDOM_STATE = 42
N_SPLITS = 5
OUTPUTS_DIR = REPO_ROOT / "outputs"
FIGURES_DIR = REPO_ROOT / "figures"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CV_SPLITTER = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
print(f"Repo root: {REPO_ROOT}")
print(f"scikit-learn: {sklearn_version}; CV: {N_SPLITS} folds; seed: {RANDOM_STATE}")

Repo root: D:\TAI_LIEU_HCMUS_K24\Năm 2 - Kì 3\Cơ sở Trí tuệ nhân tạo\Project02\Lab2_DecisionTree
scikit-learn: 1.8.0; CV: 5 folds; seed: 42


## Giao thức thí nghiệm

1. Gọi pipeline dùng chung `get_train_test()` đúng một lần; không tự split và không scale dữ liệu.
2. Sinh `ccp_alphas` từ train bằng `cost_complexity_pruning_path`; bỏ alpha cuối vì nó tạo cây tầm thường chỉ có nút gốc.
3. Đánh giá từng alpha bằng 5-fold stratified CV trên train. Chọn mean validation accuracy cao nhất; nếu hòa trong tolerance `1e-12`, ưu tiên alpha lớn hơn để có cây đơn giản hơn. Macro-F1 chỉ là chỉ số kiểm tra phụ.
4. Giữ alpha đã chọn và chạy grid `max_depth ∈ {5, 8, 10, 15}` × `min_samples_leaf ∈ {1, 5, 10, 20}` với đúng CV protocol. Nếu hòa, ưu tiên depth nhỏ hơn rồi leaf lớn hơn.
5. Fit M1 trên toàn bộ train. Chỉ sau đó mới gọi `evaluate_model()` một lần để đánh giá held-out test.

In [2]:
# Lần gọi DUY NHẤT tới shared loader trong notebook này.
X_train, X_test, y_train, y_test = get_train_test()
assert len(X_train) == len(y_train)
assert not X_train.isna().any().any()
assert set(y_train.unique()) == {"Dropout", "Enrolled", "Graduate"}
print(f"Train: {X_train.shape[0]} rows × {X_train.shape[1]} features")
print("Held-out test objects đã được niêm phong; chưa kiểm tra hoặc dùng để chọn mô hình")
display(
    y_train.value_counts().rename_axis("class").to_frame("train_count")
    .assign(train_share=lambda frame: frame["train_count"] / frame["train_count"].sum())
)

Train: 3539 rows × 90 features
Held-out test objects đã được niêm phong; chưa kiểm tra hoặc dùng để chọn mô hình


,train_count,train_share
class,,
Graduate,1767,0.499294
Dropout,1137,0.321277
Enrolled,635,0.179429


## 1. Cost-complexity pruning path trên train set

`ccp_alpha = 0` tương ứng với không cost-complexity prune. Alpha càng lớn càng phạt độ phức tạp mạnh hơn. Alpha cuối bị loại vì theo ví dụ chính thức của scikit-learn, nó tạo cây chỉ còn một nút.

In [3]:
path_estimator = DecisionTreeClassifier(random_state=RANDOM_STATE)
pruning_path = path_estimator.cost_complexity_pruning_path(X_train, y_train)
path_alphas = np.asarray(pruning_path.ccp_alphas, dtype=float)
path_impurities = np.asarray(pruning_path.impurities, dtype=float)
assert len(path_alphas) == len(path_impurities) and len(path_alphas) >= 2
assert np.all(path_alphas >= -1e-15) and np.all(np.diff(path_alphas) >= -1e-12)
candidate_alphas = np.unique(path_alphas[:-1])
assert len(candidate_alphas) > 1 and np.isclose(candidate_alphas[0], 0.0)
print(f"Pruning path: {len(path_alphas)} điểm")
print(f"Alpha duy nhất đưa vào CV: {len(candidate_alphas)}")
print(f"Khoảng alpha ứng viên: [{candidate_alphas.min():.10g}, {candidate_alphas.max():.10g}]")
print(f"Alpha bị loại (cây một nút): {path_alphas[-1]:.10g}")

Pruning path: 334 điểm
Alpha duy nhất đưa vào CV: 236
Khoảng alpha ứng viên: [0, 0.03076015029]
Alpha bị loại (cây một nút): 0.1659671558


## 2. Chọn `ccp_alpha` bằng CV — không nhìn test

Mỗi alpha được đánh giá trên cùng năm stratified folds. Bảng này cố ý không có bất kỳ test score nào.

In [4]:
alpha_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    {"ccp_alpha": candidate_alphas},
    scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
    refit=False, cv=CV_SPLITTER, n_jobs=1, return_train_score=True, error_score="raise",
)
alpha_search.fit(X_train, y_train)
alpha_results = pd.DataFrame({
    "ccp_alpha": [float(value) for value in alpha_search.cv_results_["param_ccp_alpha"]],
    "mean_train_accuracy": alpha_search.cv_results_["mean_train_accuracy"],
    "std_train_accuracy": alpha_search.cv_results_["std_train_accuracy"],
    "mean_cv_accuracy": alpha_search.cv_results_["mean_test_accuracy"],
    "std_cv_accuracy": alpha_search.cv_results_["std_test_accuracy"],
    "mean_cv_f1_macro": alpha_search.cv_results_["mean_test_f1_macro"],
    "std_cv_f1_macro": alpha_search.cv_results_["std_test_f1_macro"],
    "rank_cv_accuracy": alpha_search.cv_results_["rank_test_accuracy"],
}).sort_values("ccp_alpha", ignore_index=True)
best_alpha_score = float(alpha_results["mean_cv_accuracy"].max())
alpha_ties = alpha_results.loc[np.isclose(
    alpha_results["mean_cv_accuracy"], best_alpha_score, rtol=0.0, atol=1e-12
)].copy()
best_alpha_row = alpha_ties.sort_values("ccp_alpha", ascending=False).iloc[0]
best_alpha = float(best_alpha_row["ccp_alpha"])
print(f"Best alpha: {best_alpha:.10g}")
print(f"Mean CV accuracy: {best_alpha_score:.6f}; macro-F1: {best_alpha_row['mean_cv_f1_macro']:.6f}")
print(f"Số alpha hòa trong tolerance 1e-12: {len(alpha_ties)}")
display(alpha_results.sort_values(
    ["mean_cv_accuracy", "ccp_alpha"], ascending=[False, False]
).head(10))

Best alpha: 0.001487461358
Mean CV accuracy: 0.747669; macro-F1: 0.672329
Số alpha hòa trong tolerance 1e-12: 2


,ccp_alpha,mean_train_accuracy,std_train_accuracy,mean_cv_accuracy,std_cv_accuracy,mean_cv_f1_macro,std_cv_f1_macro,rank_cv_accuracy
211,0.001487,0.790266,0.004650,0.747669,0.012844,0.672329,0.019977,1
210,0.001375,0.798248,0.004767,0.747669,0.013802,0.675825,0.023616,2
209,0.001368,0.798743,0.004654,0.747386,0.013324,0.675900,0.023694,3
208,0.001359,0.799167,0.005403,0.747386,0.013324,0.675674,0.023558,3
207,0.001354,0.799167,0.005403,0.747386,0.013324,0.675674,0.023558,3
213,0.001605,0.786380,0.005246,0.747104,0.012666,0.675979,0.024077,6
212,0.001539,0.788782,0.006361,0.746539,0.011104,0.676166,0.023249,7
216,0.001660,0.785321,0.004641,0.746538,0.012371,0.675184,0.023765,8
215,0.001647,0.785674,0.004897,0.746538,0.012371,0.675138,0.023783,8
214,0.001635,0.785674,0.004897,0.746538,0.012371,0.675138,0.023783,8


In [5]:
alpha_curve_path = FIGURES_DIR / "C_ccp_alpha_curve.png"
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(path_alphas[:-1], path_impurities[:-1], marker="o", markersize=3, linewidth=1.2)
axes[0].axvline(best_alpha, color="tab:red", linestyle="--", label=f"selected α={best_alpha:.3g}")
axes[0].set(title="Cost-complexity path (train only)", xlabel="Effective alpha", ylabel="Total leaf impurity")
axes[0].grid(alpha=0.25); axes[0].legend()
x_values = alpha_results["ccp_alpha"].to_numpy()
train_mean = alpha_results["mean_train_accuracy"].to_numpy()
train_std = alpha_results["std_train_accuracy"].to_numpy()
cv_mean = alpha_results["mean_cv_accuracy"].to_numpy()
cv_std = alpha_results["std_cv_accuracy"].to_numpy()
axes[1].plot(x_values, train_mean, label="Mean train accuracy", color="tab:blue")
axes[1].fill_between(x_values, train_mean-train_std, train_mean+train_std, color="tab:blue", alpha=0.12)
axes[1].plot(x_values, cv_mean, label="Mean CV accuracy", color="tab:orange")
axes[1].fill_between(x_values, cv_mean-cv_std, cv_mean+cv_std, color="tab:orange", alpha=0.18)
axes[1].axvline(best_alpha, color="tab:red", linestyle="--", label=f"selected α={best_alpha:.3g}")
axes[1].scatter([best_alpha], [best_alpha_score], color="tab:red", zorder=5)
axes[1].set(title="Alpha selection by 5-fold CV (train only)", xlabel="ccp_alpha", ylabel="Accuracy")
axes[1].grid(alpha=0.25); axes[1].legend()
for axis in axes:
    axis.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
fig.suptitle("Role C — Cost-complexity pruning without test leakage", fontsize=14, fontweight="bold")
fig.tight_layout(); fig.savefig(alpha_curve_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Đã lưu: {alpha_curve_path}")

Đã lưu: D:\TAI_LIEU_HCMUS_K24\Năm 2 - Kì 3\Cơ sở Trí tuệ nhân tạo\Project02\Lab2_DecisionTree\figures\C_ccp_alpha_curve.png


## 3. Grid bắt buộc cho `max_depth` và `min_samples_leaf`

Alpha vừa chọn được giữ cố định. Grid 4 × 4 tiếp tục dùng CV trên train, nên held-out test vẫn chưa tham gia quyết định.

In [6]:
DEPTH_GRID = [5, 8, 10, 15]
LEAF_GRID = [1, 5, 10, 20]
structure_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=best_alpha),
    {"max_depth": DEPTH_GRID, "min_samples_leaf": LEAF_GRID},
    scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
    refit=False, cv=CV_SPLITTER, n_jobs=1, return_train_score=True, error_score="raise",
)
structure_search.fit(X_train, y_train)
grid_results = pd.DataFrame(structure_search.cv_results_["params"])
grid_results = grid_results.assign(
    ccp_alpha=best_alpha,
    mean_train_accuracy=structure_search.cv_results_["mean_train_accuracy"],
    std_train_accuracy=structure_search.cv_results_["std_train_accuracy"],
    mean_cv_accuracy=structure_search.cv_results_["mean_test_accuracy"],
    std_cv_accuracy=structure_search.cv_results_["std_test_accuracy"],
    mean_cv_f1_macro=structure_search.cv_results_["mean_test_f1_macro"],
    std_cv_f1_macro=structure_search.cv_results_["std_test_f1_macro"],
    rank_cv_accuracy=structure_search.cv_results_["rank_test_accuracy"],
)
grid_results[["max_depth", "min_samples_leaf"]] = grid_results[["max_depth", "min_samples_leaf"]].astype(int)
grid_results = grid_results.sort_values(
    ["rank_cv_accuracy", "max_depth", "min_samples_leaf"], ascending=[True, True, False], ignore_index=True
)
best_grid_score = float(grid_results["mean_cv_accuracy"].max())
grid_ties = grid_results.loc[np.isclose(
    grid_results["mean_cv_accuracy"], best_grid_score, rtol=0.0, atol=1e-12
)].copy()
best_grid_row = grid_ties.sort_values(
    ["max_depth", "min_samples_leaf"], ascending=[True, False]
).iloc[0]
best_max_depth = int(best_grid_row["max_depth"])
best_min_samples_leaf = int(best_grid_row["min_samples_leaf"])
assert len(grid_results) == len(DEPTH_GRID) * len(LEAF_GRID) == 16
print(
    f"Grid winner: max_depth={best_max_depth}, min_samples_leaf={best_min_samples_leaf}, "
    f"mean CV accuracy={best_grid_score:.6f}, macro-F1={best_grid_row['mean_cv_f1_macro']:.6f}"
)
print(f"Số cấu hình hòa trong tolerance 1e-12: {len(grid_ties)}")
display(grid_results)

Grid winner: max_depth=5, min_samples_leaf=20, mean CV accuracy=0.748235, macro-F1=0.654465
Số cấu hình hòa trong tolerance 1e-12: 1


,max_depth,min_samples_leaf,ccp_alpha,mean_train_accuracy,std_train_accuracy,mean_cv_accuracy,std_cv_accuracy,mean_cv_f1_macro,std_cv_f1_macro,rank_cv_accuracy
0,5,20,0.001487,0.762221,0.002894,0.748235,0.015124,0.654465,0.031242,1
1,10,1,0.001487,0.790266,0.004650,0.747951,0.012594,0.672854,0.020036,2
2,15,1,0.001487,0.790266,0.004650,0.747669,0.012844,0.672329,0.019977,3
3,10,5,0.001487,0.789630,0.004847,0.747669,0.012075,0.672378,0.019491,4
4,15,5,0.001487,0.789630,0.004847,0.747669,0.012075,0.672378,0.019491,4
5,10,10,0.001487,0.784614,0.005212,0.747387,0.012577,0.672609,0.018657,6
6,15,10,0.001487,0.784614,0.005212,0.747387,0.012577,0.672609,0.018657,6
7,8,5,0.001487,0.787016,0.006499,0.747386,0.010056,0.670261,0.018985,8
8,8,10,0.001487,0.782495,0.006674,0.747104,0.010636,0.669713,0.017889,9
9,8,1,0.001487,0.787652,0.006414,0.747103,0.011060,0.669665,0.019289,10


## 4. Khóa cấu hình và fit M1 trên toàn bộ train set

Từ thời điểm này, `ccp_alpha`, `max_depth` và `min_samples_leaf` không còn được thay đổi. Cell kế tiếp chưa tính test metric.

In [7]:
m1 = DecisionTreeClassifier(
    random_state=RANDOM_STATE, ccp_alpha=best_alpha,
    max_depth=best_max_depth, min_samples_leaf=best_min_samples_leaf,
)
m1.fit(X_train, y_train)
m1_params = {
    "ccp_alpha": float(best_alpha),
    "max_depth": best_max_depth,
    "min_samples_leaf": best_min_samples_leaf,
    "random_state": RANDOM_STATE,
}
assert m1.get_params()["ccp_alpha"] == best_alpha
assert m1.get_params()["max_depth"] == best_max_depth
assert m1.get_params()["min_samples_leaf"] == best_min_samples_leaf
print(f"Cấu hình M1 đã khóa: {m1_params}")
print(f"Cây fit trên full train: depth={m1.get_depth()}, leaves={m1.get_n_leaves()}")

Cấu hình M1 đã khóa: {'ccp_alpha': 0.0014874613584826332, 'max_depth': 5, 'min_samples_leaf': 20, 'random_state': 42}
Cây fit trên full train: depth=5, leaves=17


## 5. Đánh giá held-out test đúng một lần qua helper dùng chung

Đây là điểm đầu tiên dùng test labels để tính hiệu năng. Một lời gọi `evaluate_model()` tính đủ schema 16 cột, append idempotent M1, xuất classification report và confusion matrix.

In [8]:
# Chỉ từ đây mới kiểm tra và dùng held-out test, sau khi m1_params đã khóa.
assert len(X_test) == len(y_test)
assert X_train.columns.equals(X_test.columns)
assert not X_test.isna().any().any()
assert set(y_test.unique()) == set(m1.classes_)
classification_report_path = OUTPUTS_DIR / "classification_report_M1.txt"
confusion_matrix_path = FIGURES_DIR / "C_cm_M1.png"
m1_result = evaluate_model(
    m1, X_train, y_train, X_test, y_test,
    model_id="M1", model_name="Cost-complexity pruned tree",
    params=m1_params, author="C",
    classification_report_path=classification_report_path,
    confusion_matrix_path=confusion_matrix_path,
)
display(pd.DataFrame([m1_result])[RESULT_COLUMNS])

,model_id,model_name,params,train_acc,test_acc,error_rate,precision_macro,recall_macro,f1_macro,roc_auc_macro,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves,author
0,M1,Cost-complexity pruned tree,"{""ccp_alpha"":0.0014874613584826332,""max_depth""...",0.768014,0.755932,0.244068,0.710494,0.660165,0.672925,0.847692,0.686620,0.345912,0.947964,5,17,C


In [9]:
tree_path = plot_tree_figure(
    m1, X_train.columns, m1.classes_, FIGURES_DIR / "C_tree_M1.png",
    title=(
        "M1 — Cost-complexity pruned decision tree "
        f"(α={best_alpha:.4g}, depth={m1.get_depth()}, leaves={m1.get_n_leaves()})"
    ),
)
print(f"Đã lưu cây M1: {tree_path}")
print(f"Đã lưu confusion matrix: {confusion_matrix_path}")
print(f"Đã lưu classification report: {classification_report_path}")

Đã lưu cây M1: D:\TAI_LIEU_HCMUS_K24\Năm 2 - Kì 3\Cơ sở Trí tuệ nhân tạo\Project02\Lab2_DecisionTree\figures\C_tree_M1.png
Đã lưu confusion matrix: D:\TAI_LIEU_HCMUS_K24\Năm 2 - Kì 3\Cơ sở Trí tuệ nhân tạo\Project02\Lab2_DecisionTree\figures\C_cm_M1.png
Đã lưu classification report: D:\TAI_LIEU_HCMUS_K24\Năm 2 - Kì 3\Cơ sở Trí tuệ nhân tạo\Project02\Lab2_DecisionTree\outputs\classification_report_M1.txt


## 6. So sánh M0–M1 và diễn giải

M0 được đọc từ artifact chung của Role B; notebook không huấn luyện lại M0 và không dùng M0 test score để chọn M1. So sánh chỉ diễn ra sau khi M1 đã đánh giá test.

In [10]:
results_path = OUTPUTS_DIR / "results.csv"
results_df = pd.read_csv(results_path)
comparison = results_df.loc[results_df["model_id"].isin(["M0", "M1"])].copy()
assert comparison["model_id"].value_counts().to_dict() == {"M0": 1, "M1": 1}
comparison = comparison.set_index("model_id").loc[["M0", "M1"]]
comparison_columns = [
    "train_acc", "test_acc", "error_rate", "precision_macro",
    "recall_macro", "f1_macro", "roc_auc_macro",
    "recall_dropout", "recall_enrolled", "recall_graduate",
    "tree_depth", "n_leaves",
]
display(comparison[comparison_columns])
m0_row, m1_row = comparison.loc["M0"], comparison.loc["M1"]
summary = pd.Series({
    "test_accuracy_delta_M1_minus_M0": m1_row["test_acc"] - m0_row["test_acc"],
    "macro_f1_delta_M1_minus_M0": m1_row["f1_macro"] - m0_row["f1_macro"],
    "depth_reduction_fraction": 1.0 - m1_row["tree_depth"] / m0_row["tree_depth"],
    "leaf_reduction_fraction": 1.0 - m1_row["n_leaves"] / m0_row["n_leaves"],
    "generalization_gap_M0": m0_row["train_acc"] - m0_row["test_acc"],
    "generalization_gap_M1": m1_row["train_acc"] - m1_row["test_acc"],
}, name="value")
display(summary.to_frame())

,train_acc,test_acc,error_rate,precision_macro,recall_macro,f1_macro,roc_auc_macro,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves
model_id,,,,,,,,,,,,
M0,1.000000,0.668927,0.331073,0.607642,0.609310,0.608271,0.719456,0.679577,0.383648,0.764706,27,634
M1,0.768014,0.755932,0.244068,0.710494,0.660165,0.672925,0.847692,0.686620,0.345912,0.947964,5,17


,value
test_accuracy_delta_M1_minus_M0,0.087006
macro_f1_delta_M1_minus_M0,0.064654
depth_reduction_fraction,0.814815
leaf_reduction_fraction,0.973186
generalization_gap_M0,0.331073
generalization_gap_M1,0.012081


In [11]:
accuracy_delta = float(summary["test_accuracy_delta_M1_minus_M0"])
leaf_reduction = float(summary["leaf_reduction_fraction"])
depth_reduction = float(summary["depth_reduction_fraction"])
gap_reduction = float(summary["generalization_gap_M0"] - summary["generalization_gap_M1"])
print(f"M1: α={best_alpha:.10g}, max_depth={best_max_depth}, min_samples_leaf={best_min_samples_leaf}.")
print(
    f"So với M0, depth giảm {depth_reduction:.1%}, số lá giảm {leaf_reduction:.1%}; "
    f"test accuracy thay đổi {accuracy_delta:+.4f} "
    f"({accuracy_delta * 100:+.2f} điểm phần trăm)."
)
print(
    f"Generalization gap giảm {gap_reduction:.4f}, từ "
    f"{summary['generalization_gap_M0']:.4f} xuống {summary['generalization_gap_M1']:.4f}."
)
print(
    "Cơ chế: pruning loại các nhánh có mức giảm impurity không đủ bù chi phí độ phức tạp; "
    "depth/leaf constraints tiếp tục tránh các phân vùng quá nhỏ. Điều này giảm variance và "
    "train–test gap, đổi lại có thể tăng bias. CV quyết định điểm cân bằng trước khi mở test."
)
print("\nClassification report M1:\n")
print(classification_report_path.read_text(encoding="utf-8"))

M1: α=0.001487461358, max_depth=5, min_samples_leaf=20.
So với M0, depth giảm 81.5%, số lá giảm 97.3%; test accuracy thay đổi +0.0870 (+8.70 điểm phần trăm).
Generalization gap giảm 0.3190, từ 0.3311 xuống 0.0121.
Cơ chế: pruning loại các nhánh có mức giảm impurity không đủ bù chi phí độ phức tạp; depth/leaf constraints tiếp tục tránh các phân vùng quá nhỏ. Điều này giảm variance và train–test gap, đổi lại có thể tăng bias. CV quyết định điểm cân bằng trước khi mở test.

Classification report M1:

              precision    recall  f1-score   support

     Dropout     0.8263    0.6866    0.7500       284
    Enrolled     0.5392    0.3459    0.4215       159
    Graduate     0.7660    0.9480    0.8473       442

    accuracy                         0.7559       885
   macro avg     0.7105    0.6602    0.6729       885
weighted avg     0.7446    0.7559    0.7396       885



## 7. Quality gates bàn giao

Các assertion kiểm tra schema chung, M1 duy nhất, quan hệ metric, grid bắt buộc, cấu hình fitted model và toàn bộ artifact Role C.

In [12]:
final_results = pd.read_csv(results_path)
assert final_results.columns.tolist() == RESULT_COLUMNS
assert (final_results["model_id"] == "M1").sum() == 1
assert m1_result["model_id"] == "M1" and m1_result["author"] == "C"
assert np.isclose(m1_result["error_rate"], 1.0 - m1_result["test_acc"])
assert len(grid_results) == 16
assert set(grid_results["max_depth"]) == set(DEPTH_GRID)
assert set(grid_results["min_samples_leaf"]) == set(LEAF_GRID)
assert m1.get_depth() <= best_max_depth
assert m1.get_n_leaves() < int(m0_row["n_leaves"])
assert m1.get_depth() < int(m0_row["tree_depth"])
required_artifacts = [
    alpha_curve_path, tree_path, confusion_matrix_path, classification_report_path, results_path
]
missing_or_empty = [
    str(path) for path in required_artifacts
    if not path.is_file() or path.stat().st_size == 0
]
assert not missing_or_empty, f"Artifact thiếu/rỗng: {missing_or_empty}"
print("PASS — M1 duy nhất, schema 16 cột đúng, grid đủ 16 cấu hình.")
print("PASS — M1 nhỏ hơn M0 và toàn bộ artifact Role C tồn tại, không rỗng.")
print("PASS — Test chỉ được đánh giá sau khi cấu hình M1 đã khóa.")

PASS — M1 duy nhất, schema 16 cột đúng, grid đủ 16 cấu hình.
PASS — M1 nhỏ hơn M0 và toàn bộ artifact Role C tồn tại, không rỗng.
PASS — Test chỉ được đánh giá sau khi cấu hình M1 đã khóa.


## Kết luận và bước bàn giao

Kết quả số trong các cell trên là nguồn duy nhất để viết mục **Improvement Method 1**. Trước bàn giao: Restart & Run All, validate cấu trúc, kiểm tra hình ảnh và đối chiếu độc lập dòng M1 trong `outputs/results.csv`.